# 🐯 TIGeR — Corrected Baseline Run

Every repair-side number currently in `paper_assets/` was produced by a
measurement harness that was broken in four independent ways. This notebook
re-runs the pipeline on the fixed branch (`docs/fixes-backlog-audit`) so the
tables can be regenerated from an instrument that works.

**What was wrong, and why the numbers must move**

| # | Defect | Effect on the published tables |
|---|---|---|
| A1 | The "No Gamma Gate" ablation wrote `cfg["fusion"]["gamma"]`; the router reads `cfg["arbiter"]["gamma"]` | That row ran the *identical* configuration as Full System. The two matched because they were the same run. §7.5 of `paper_draft_materials.md` analyses that identity as a finding. |
| A3 / A3b | The random baseline emitted an `E4` class the router silently relabelled as E3, could never choose CLEAN, and was unseeded | The 3.2% denominator behind the "+49.4%" headline was neither uniform nor reproducible. |
| A5 | "Restoration Accuracy" scored `field == "color"` only | Material and pattern patches were committed but never scored; image repairs were never scored at all. |
| A8 | Per-signal precision counted any dirty row as a hit | The 0.85 precision floor certified "fires on dirty rows", not "identifies the error it names". |
| A6 | `t2v_policy.allowed_categories` listed fashion categories only | Every ABO image repair was force-escalated. Cross-domain T2V was never exercised. |
| A7 | Fusion was never loaded by `detect` or `repair` | The advertised P=0.888 was an offline row; the live pipeline ran at P=0.793. |
| D5–D7 | Five colours were both domain values and aliases of other values | Probes scored duplicates as rivals; the verifier vetoed correct repairs; `synthgen` crashed on ~1/3 of products. |

**Runtime:** roughly 25–40 min on a T4. Turn the GPU on
(*Notebook options → Accelerator → GPU T4 x2*) before starting.

## Step 0 · Clone the fixed branch and install

In [ ]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone -q -b docs/fixes-backlog-audit https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!git log --oneline -1

# H7: never use --force-reinstall here. It rebuilds the whole dependency tree,
# pulls an incompatible torchvision, and every later cell dies with
# "operator torchvision::nms does not exist". Clear only the local package.
!pip uninstall -y tiger -q 2>/dev/null
!pip install --no-cache-dir -e ".[dev,vlm,gen]" -q
print("\n✅ installed")

## Step 1 · Guard: confirm the fixes are actually present

This is cheap and it fails loudly. Without it, cloning the wrong branch wastes
the entire run and produces numbers that look plausible and are not.

In [ ]:
import inspect
from tiger import cli
from tiger.schema import Schema, load_schema
from tiger.fusion import PROBE_TARGETS
from tiger.eval import repair_ablation as RA

checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

src = inspect.getsource(RA.run_repair_ablations)
check("A1  gamma written to cfg['arbiter']",
      'cfg_no_gamma.setdefault("arbiter", {})["gamma"] = 0.0' in src)
check("A4  configs deep-copied", "copy.deepcopy(cfg)" in src)
check("A3  baseline emits no E4 key",
      '"E4"' not in inspect.getsource(RA.DummyArbiter.predict_proba))
check("A3b baseline is seeded",
      "seed" in inspect.signature(RA.DummyArbiter.__init__).parameters)
check("A5  every audited field scored", hasattr(RA, "truth_from_audit"))
check("A5  image repairs scored", hasattr(RA, "image_provenance"))
check("A7  --fusion flag exposed", '"--fusion"' in inspect.getsource(cli.main))
check("A8  probes scored on-target", isinstance(PROBE_TARGETS, dict) and "color" in PROBE_TARGETS)
check("D5  surface_forms available", hasattr(Schema, "surface_forms"))

schema = load_schema("configs/schema.yaml")
dom = schema.domain("color")
check("D5  no duplicate colours in the domain",
      len(dom) == len({schema.normalize("color", v) for v in dom}),
      f"{len(dom)} entries: {dom}")

from tiger.data.synthgen import COLOR_RGB
missing = [c for c in dom if c != "multicolour" and c not in COLOR_RGB]
check("D5  synthgen can render every colour", not missing, f"missing: {missing}")

cfg = cli.load_cfg()
allowed = cfg["arbiter"]["t2v_policy"]["allowed_categories"]
check("A6  ABO verticals in the T2V allowlist",
      "electronics" in allowed, f"allowed: {allowed}")

width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:<{width}}  {detail}")

failed = [n for n, ok, _ in checks if not ok]
assert not failed, f"\n\nWRONG BRANCH OR STALE INSTALL. Missing: {failed}"
print("\n✅ all fixes present — safe to proceed")

## Step 2 · Run the test suite

114 tests. If any fail, stop: the environment is not what the fixes were verified against.

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## Step 3 · Pin γ and record it with the run

γ has had four different values across the repo (0.40 in the config, 0.60 in
the docs and the ABO analysis, 0.85 in `paper_concepts.md`, 0.448 recalibrated).
Nobody can say which produced which published number — that is finding C1, and
it happened because recalibration edits `configs/tiger.yaml` in place with no
record.

Set it **once**, here, and write the effective values into the outputs so this
run is self-documenting.

`GAMMA = 0.60` reproduces the operating point the Fashion tables were built at
(the committed confidence histogram is captioned "Gamma threshold (0.6)").
Change it only deliberately.

In [ ]:
import json, re, yaml
from pathlib import Path

GAMMA = 0.60          # operating point for THIS run
NOISE_SEED = 7
BASELINE_SEED = 42    # seeds the random-routing baseline (A3b)

cfg_path = Path("configs/tiger.yaml")
text = cfg_path.read_text()

# Surgical line edit, NOT yaml.safe_dump: dumping would round-trip the file and
# silently delete every comment in it, including the rationale for the T2V
# allowlist and the noise rates. Config comments are the only place several of
# these decisions are recorded.
# Replace the whole line including its trailing comment. Leaving the old
# comment behind ("lowered from 0.60 ...") next to a value of 0.60 is precisely
# the config/doc drift that produced finding C1 in the first place.
text, n = re.subn(
    r"^(\s*)gamma:.*$",
    rf"\g<1>gamma: {GAMMA}  # Eq. 22 confidence gate; pinned by tiger_corrected_run.ipynb",
    text, count=1, flags=re.M)
assert n == 1, "could not find `gamma:` in configs/tiger.yaml"

if not re.search(r"^eval:", text, flags=re.M):
    text += ("\neval:\n"
             "  # Seeds the random-routing baseline so the 'No Arbiter' row is\n"
             "  # reproducible (finding A3b).\n"
             f"  random_baseline_seed: {BASELINE_SEED}\n")
else:
    text, n = re.subn(r"^(\s*random_baseline_seed:\s*)\d+", rf"\g<1>{BASELINE_SEED}",
                      text, count=1, flags=re.M)
    assert n == 1
cfg_path.write_text(text)

cfg = yaml.safe_load(cfg_path.read_text())
assert cfg["arbiter"]["gamma"] == GAMMA
assert cfg["eval"]["random_baseline_seed"] == BASELINE_SEED

manifest = {
    "branch": "docs/fixes-backlog-audit",
    "gamma": GAMMA,
    "noise_seed": NOISE_SEED,
    "random_baseline_seed": BASELINE_SEED,
    "dismiss_threshold": cfg["arbiter"]["dismiss_threshold"],
    "t2v_allowed_categories": cfg["arbiter"]["t2v_policy"]["allowed_categories"],
    "precision_floor": cfg["sieve"]["precision_floor"],
    "clip_model": cfg["models"]["clip_model_name"],
    "independent_verifier": cfg["models"]["independent_verifier"],
    "fusion_applied": False,
}
Path("data/outputs").mkdir(parents=True, exist_ok=True)
Path("data/outputs/run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


---
# Phase A · Synthetic catalogue → detection numbers

Regenerates the detection table (precision / recall / F1 and per-error-type
recall). `synthgen` has been unbuildable since the ABO schema edit — D5 is what
makes this step run at all.

In [ ]:
!python -m tiger.cli synthgen

In [ ]:
!python -m tiger.cli calibrate

**`train-arbiter` is the slow one** — it runs eight noise seeds end to end. Expect several minutes.

In [ ]:
!python -m tiger.cli train-arbiter

In [ ]:
!python -m tiger.cli calibrate-fusion

### Detection sweep and ablations

`calibrate-fusion` output above is worth reading closely: A8 changed how
per-signal precision is scored, so a signal that previously cleared the 0.85
floor may now be quarantined. That is the correction working.

In [ ]:
!python -m tiger.cli sweep --seeds 7,8,9,10,11

In [ ]:
!python -m tiger.cli ablate --seeds 7,8,9,10,11

### Archive Phase A before Fashion overwrites `data/sample/`

In [ ]:
!mkdir -p /kaggle/working/phaseA_synthetic
!cp -r data/outputs data/thresholds /kaggle/working/phaseA_synthetic/ 2>/dev/null
!ls /kaggle/working/phaseA_synthetic/outputs | head -20
print("\n✅ Phase A archived")

---
# Phase B · Fashion catalogue → repair numbers

This is where the tables that need correcting come from.

⚠️ `import-fashion` **overwrites `data/sample/`**. Phase A is archived above.

Add the dataset first: *Add Input* → search **"Fashion Product Images Dataset"**
by `paramaggarwal`.

In [ ]:
from pathlib import Path

# H8/H9: Kaggle's mount layout has moved before. Find it rather than hardcode it.
roots = [p for p in Path("/kaggle/input").rglob("styles.csv")]
assert roots, "Fashion dataset not attached — use 'Add Input' in the right sidebar."
FASHION_DIR = roots[0].parent
print("fashion dataset:", FASHION_DIR)

In [ ]:
!python -m tiger.cli import-fashion --source {FASHION_DIR}

Recalibrate everything on the Fashion distribution — thresholds, router and fusion are all domain-specific.

In [ ]:
!python -m tiger.cli calibrate

In [ ]:
!python -m tiger.cli train-arbiter

In [ ]:
!python -m tiger.cli calibrate-fusion

In [ ]:
!python -m tiger.cli noise --seed 7

### The corrected repair ablation

`--independent` uses SigLIP, which is the verifier the paper reports.
`--vlm-judge` (Gemini) is deliberately **not** used: its model ID is unverified
(finding A2), and a bad ID now raises rather than silently vetoing every repair.

Fusion is left off, matching the operating point the published repair numbers
were produced under. Add `--fusion` for the alternative operating point once
you have the default to compare against.

In [ ]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

## Results

Two columns are new (A5). `Attr Accuracy` covers colour, material and pattern
instead of colour alone; `T2V Accuracy` scores image repairs, which were
previously not scored at all. Every metric now carries its own N — the old
headline was 52.6% on n=19 and the denominator appeared only in a LaTeX caption.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 50)

df = pd.read_csv("data/outputs/repair_ablations_summary.csv")
cols = [c for c in ["Configuration", "Repaired", "Escalated", "Total Attempted",
                    "Attr Accuracy", "Attr Cases", "T2V Accuracy", "T2V Cases",
                    "Color Accuracy", "V2T Cases"] if c in df.columns]
display(df[cols])

full = df[df["Configuration"] == "Full System"]
nog  = df[df["Configuration"].str.contains("Gamma", na=False)]
if len(full) and len(nog):
    same = (full["Repaired"].iat[0] == nog["Repaired"].iat[0]
            and full["Escalated"].iat[0] == nog["Escalated"].iat[0])
    print("\nGamma gate row vs Full System:",
          "STILL IDENTICAL — investigate, A1 should have separated these" if same
          else "different ✅ — the gate is now genuinely ablated")

### V2T estimator attribution

Every colour repair records which estimator supplied the value (the HSV pixel
histogram or the CLIP probe) and whether the *other* one would have been right.

This report decides what to work on next: if `always pixel` far exceeds
`always probe`, the encoder is the bottleneck (B7); if the reverse, the colour
estimator is (B1–B5, currently blocked on having this data locally). The
`agree / disagree` split sizes what an abstention gate would buy — that is the
parked ⚑ item B6.

In [ ]:
import pandas as pd
from pathlib import Path

p = Path("data/outputs/v2t_estimator_diagnostics.csv")
if p.exists():
    d = pd.read_csv(p)
    display(d.head(30))
    print("\nrows:", len(d))
else:
    print("No scored V2T cases in this run — nothing to attribute.")

## Export everything

In [ ]:
!cp -r data/outputs data/thresholds data/processed /kaggle/working/ 2>/dev/null
!cd /kaggle/working && zip -rq tiger_corrected_run.zip outputs thresholds processed phaseA_synthetic
!ls -la /kaggle/working/tiger_corrected_run.zip
print("\n✅ Download tiger_corrected_run.zip from the right sidebar.")

---
## What to do with these numbers

1. **Compare against `paper_assets/paper_draft_materials.md` §1.** Expect movement.
   Four independent defects fed that table; none of them biased it in a
   predictable direction, so do not assume the corrected numbers are worse.

2. **§7.5 and `ROADMAP_PROGRESS.md` H11 must be rewritten or withdrawn** (E3).
   They explain why Full System and No-Gamma-Gate matched. They matched because
   A1 made them the same run. If the corrected run still shows overlap, prove it
   properly by intersecting the escalated `row_id` sets — identical totals are
   not identical sets.

3. **`reviewer_defense.md` Attack 3 is the urgent one** (E2). It presents the
   47.4% as "safely escalated to a human". They were not: that figure is error
   among rows the system *repaired and committed*. Escalations are the separate
   269. Fix the claim before anyone reads it.

4. **Report N with every accuracy figure.** The CSV now carries them.

5. **Then run the ABO notebook** on this branch. A6 opened the T2V path that was
   closed for the entire cross-domain evaluation, so RQ3's image-repair evidence
   does not exist yet.

Still open and needing you specifically:
- **A2** — pin a Gemini model ID with a live key (`genai.list_models()`), and
  determine whether any published number came from a `--vlm-judge` run.
- **⚑ B6, ⚑ D4** — parked design decisions, not bugs.